# Laboratorio 02 — JOINs libres con PySpark

**Semana:** 02 | **Actividad de referencia:** Actividad 02  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica los JOINs aprendidos en la Actividad 02 sobre un dataset **de tu elección que tenga relaciones entre tablas** (al menos 2 archivos que se puedan unir por una llave común). El dataset puede ser el mismo que usaste en el Lab 01 si ya tiene múltiples tablas relacionadas.

Entrega esperada: este notebook completo con todas las celdas ejecutadas y las celdas markdown respondidas.

## Parte 1 — Descripción del dataset

Documenta tu dataset antes de escribir código:

1. **Nombre y fuente:** ¿Cómo se llama el dataset y de dónde lo obtuviste? (incluye URL)
2. **Dominio:** ¿Qué problema o área describe?
3. **¿Por qué lo elegiste?** ¿Qué pregunta de negocio quieres responder?
4. **Tablas disponibles:** Lista cada archivo/tabla con su llave primaria y foránea:

| Tabla | Llave primaria | Llave foránea hacia |
|-------|----------------|---------------------|
| ... | ... | ... |

5. **Diagrama de relaciones** (en texto o ASCII):

6. **Preguntas de negocio:** Lista al menos 3 preguntas que requieran combinar tablas para responderse.

> Fuentes sugeridas si necesitas un dataset relacional:
> - [Kaggle](https://www.kaggle.com/datasets) → busca datasets con múltiples archivos
> - [MovieLens](https://grouplens.org/datasets/movielens/) — movies + ratings + tags
> - [datos.gob.es](https://datos.gob.es)

**Escribe tu respuesta aquí:**

## Parte 2 — Carga y perfil técnico de cada tabla

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan, sum as spark_sum

VOL = "/Volumes/workspace/default/week_2"  # ajusta si usas un volumen distinto

# Carga cada tabla — ajusta nombres y formato según tu dataset
df_tabla1 = spark.read.format("csv").option("header", "true").option("inferSchema", "true") \
    .load(f"{VOL}/tabla1.csv")
df_tabla2 = spark.read.format("csv").option("header", "true").option("inferSchema", "true") \
    .load(f"{VOL}/tabla2.csv")

for nombre, df in [("tabla1", df_tabla1), ("tabla2", df_tabla2)]:
    print(f"\n--- {nombre} ---")
    print(f"Filas: {df.count():,} | Columnas: {len(df.columns)}")
    df.printSchema()

In [ ]:
# Estadísticas descriptivas por tabla
print("=== Tabla 1 ===")
df_tabla1.describe().show(truncate=False)

print("=== Tabla 2 ===")
df_tabla2.describe().show(truncate=False)

In [ ]:
# Nulos y vacíos por tabla
def perfil_nulos(df, nombre):
    total = df.count()
    numeric_types = {"double", "float", "long", "integer", "short", "byte"}
    nulos = df.select([
        spark_sum(
            when(
                col(c).isNull() |
                (isnan(col(c)) if df.schema[c].dataType.typeName() in numeric_types else F.lit(False)) |
                (col(c).cast("string") == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    
    print(f"\n--- Nulos en {nombre} ({total:,} filas) ---")
    print(f"{'Columna':<35} {'Nulos':>8} {'%':>8}")
    print("-" * 54)
    for c, n in sorted(nulos.items(), key=lambda x: -x[1]):
        if n > 0:
            print(f"{c:<35} {n:>8,} {n/total*100:>7.1f}%")
    if all(v == 0 for v in nulos.values()):
        print("  Sin nulos detectados")

perfil_nulos(df_tabla1, "tabla1")
perfil_nulos(df_tabla2, "tabla2")

In [ ]:
# Verificar duplicados en las llaves primarias
# Ajusta el nombre de la columna llave
LLAVE_T1 = "id"  # cambia por la llave real de tabla1
LLAVE_T2 = "id"  # cambia por la llave real de tabla2

dup_t1 = df_tabla1.groupBy(LLAVE_T1).count().filter(F.col("count") > 1).count()
dup_t2 = df_tabla2.groupBy(LLAVE_T2).count().filter(F.col("count") > 1).count()

print(f"Duplicados en llave de tabla1: {dup_t1:,}")
print(f"Duplicados en llave de tabla2: {dup_t2:,}")

**Observaciones del perfil técnico:** ¿Los tipos de las llaves coinciden entre tablas? ¿Hay nulos en columnas clave? ¿Hay duplicados en las llaves primarias? ¿Qué implica eso para los JOINs?

## Parte 3 — Transformaciones previas al JOIN

Estandariza las llaves y limpia los datos antes de unir. Aplica al menos:
- Cast de tipos si las llaves no coinciden
- Renombrado de columnas ambiguas
- Cualquier limpieza necesaria (regexp_replace, trim, etc.)

Explica en markdown cada decisión.

In [ ]:
# Limpieza y preparación antes del JOIN

**Explicación de las transformaciones previas:**

## Parte 4 — JOINs con análisis de impacto

Implementa al menos **3 tipos de JOIN distintos** sobre tus tablas. Para cada uno, documenta cuántos registros tenías antes y cuántos quedaron después, y explica por qué.

In [ ]:
# JOIN 1 — INNER JOIN
# Documenta qué tablas unes y por qué INNER tiene sentido aquí

**Análisis JOIN 1:** ¿Cuántos registros se perdieron? ¿Por qué? ¿Era esperado?

In [ ]:
# JOIN 2 — LEFT JOIN
# ¿Por qué LEFT en vez de INNER aquí?

**Análisis JOIN 2:** ¿Cuántos registros tienen null en columnas de la tabla derecha? ¿Qué significa eso en el contexto de tu dataset?

In [ ]:
# JOIN 3 — Anti JOIN (left_anti)
# ¿Qué registros de la izquierda NO tienen pareja en la derecha?

**Análisis Anti JOIN:** ¿Qué te dice este resultado sobre la calidad o integridad del modelo de datos?

## Parte 5 — Análisis de negocio sobre el DataFrame unido

Con el DataFrame resultante de los JOINs, responde las 3 preguntas de negocio que planteaste en la Parte 1. Cada respuesta debe tener:
- Un bloque de código PySpark
- Una celda markdown con tu conclusión en lenguaje natural

In [ ]:
# Pregunta 1:

**Conclusión pregunta 1:**

In [ ]:
# Pregunta 2:

**Conclusión pregunta 2:**

In [ ]:
# Pregunta 3:

**Conclusión pregunta 3:**

## Parte 6 — Reflexión final

Responde en esta celda:

1. ¿Qué tipo de JOIN fue el más revelador para entender tu dataset?
2. ¿Encontraste algún problema de integridad referencial (registros que no tienen pareja)? ¿Qué implica?
3. ¿Qué columna del DataFrame unido resultó ser la más útil para el análisis?
4. Si fueras a construir un pipeline de producción con este dataset, ¿qué validaciones harías antes de los JOINs?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_02/laboratorios/lab_02_joins.ipynb semana_02/laboratorios/<tu-nombre>/lab_02_joins.ipynb

git add semana_02/laboratorios/<tu-nombre>/lab_02_joins.ipynb
git commit -m "lab: semana02 lab02 joins <nombre-dataset> - <tu-nombre>"
git push origin develop
```